<a href="https://colab.research.google.com/github/matiasbelsito7/predictor_resultados_mundial_2026/blob/main/flashscore_scraper_mundial_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌍 Flashscore Scraper - Mundial 2026
## Descarga automática de estadísticas de las 48 selecciones

Este notebook descarga los últimos 40 partidos de cada selección clasificada al Mundial 2026,
incluyendo estadísticas generales y por jugador.

### ¿Cuánto tarda?
- **1 selección:** ~3-5 minutos
- **48 selecciones:** ~3-4 horas (se puede pausar y retomar)

### ¿Qué datos descarga?
- Resultado, fecha, competición
- Estadísticas del partido: posesión, tiros, corners, tarjetas, etc.
- Estadísticas individuales: goles, asistencias, minutos, etc.

---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 📦 PASO 1: Instalar dependencias
Ejecutar solo una vez al inicio de la sesión.

In [ ]:
# Instalar Playwright y sus dependencias del sistema
!pip install playwright -q
!apt-get install -y chromium-browser chromium-chromedriver > /dev/null 2>&1
!playwright install chromium
!playwright install-deps chromium
print('✅ Dependencias instaladas correctamente')

Installing dependencies...
Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-freefont-ttf is already the newest version (20120503-10build1

## ⚙️ PASO 2: Configuración e imports

In [ ]:
# ─── GOOGLE DRIVE ─────────────────────────────────────────────────────────────
from google.colab import drive
import asyncio
import csv
import json
import os
import time
import re
from datetime import datetime
from playwright.async_api import async_playwright
from pathlib import Path
drive.mount('/content/drive')

# ─── CONFIGURACIÓN ────────────────────────────────────────────────────────────
PARTIDOS_POR_SELECCION = 20
ESPERA_ENTRE_SELECCIONES = 5
ESPERA_ENTRE_PARTIDOS = 2

OUTPUT_DIR = '/content/drive/MyDrive/flashscore_data'
LOG_FILE = f'{OUTPUT_DIR}/progreso.json'

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f'📁 Carpeta de datos: {OUTPUT_DIR}')
print(f'📄 Archivo de progreso: {LOG_FILE}')
print(f'⚽ Partidos a descargar por selección: {PARTIDOS_POR_SELECCION}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Carpeta de datos: /content/drive/MyDrive/flashscore_data
📄 Archivo de progreso: /content/drive/MyDrive/flashscore_data/progreso.json
⚽ Partidos a descargar por selección: 20


## 🌍 PASO 3: Lista de las 48 selecciones del Mundial 2026

Cada selección tiene un ID único en Flashscore.
El formato es: `https://www.flashscore.com/team/NOMBRE/ID/results/`

In [ ]:
# ─── SELECCIONES DEL MUNDIAL 2026 con sus IDs de Flashscore ───────────────────
# Formato: (nombre_archivo, nombre_flashscore, id_flashscore)

SELECCIONES = [

    # CONMEBOL (6)
    ('argentina',       'argentina',         'f9OppQjp'),
    ('brasil',          'brazil',            'I9l9aqLq'),
    ('colombia',        'colombia',          'G02s4PCS'),
    ('uruguay',         'uruguay',           'xMk44orG'),
    ('ecuador',         'ecuador',           '8tbm8Tri'),
    ('paraguay',        'paraguay',          'YaNlqp6j'),

    # UEFA (16)
    ('alemania',        'germany',           'ptQide1O'),
    ('espana',          'spain',             'bLyo6mco'),
    ('francia',         'france',            'QkGeVG1n'),
    ('portugal',        'portugal',          'WvJrjFVN'),
    ('inglaterra',      'england',           'j9N9ZNFA'),
    ('belgica',         'belgium',           'GbB957na'),
    ('netherlands',     'netherlands',       'WYintcWb'),
    ('croacia',         'croatia',           'K8aznggo'),
    ('austria',         'austria',           'naHiWdnt'),
    ('suiza',           'switzerland',       'rHJ2vy1B'),
    ('escocia',         'scotland',          'fZRU25WH'),
    ('turquia',         'turkey',            'QeijuHo5'),
    ('republica_checa', 'czech-republic',    '6LHwBDGU'),

    ('bosnia',          'bosnia-herzegovina','fqe7WYTr'),
    ('noruega',         'norway',            '8rP6JO0H'),
    ('suecia',          'sweden',            'OQyqbHWB'),

    # CONCACAF (6)
    ('estados_unidos',  'usa',               'fuitL4CF'),
    ('mexico',          'mexico',            'O6iHcNkd'),
    ('canada',          'canada',            'x4toKORL'),
    ('panama',          'panama',            'OWKqbCfi'),

    ('curazao',         'curacao',           'bLLGpOkQ'),
    ('haiti',           'haiti',             'nk4v10Z1'),

    # AFC (9)
    ('japon',           'japan',             'ULXPdOUj'),
    ('corea_sur',       'south-korea',       'K6Gs7P6G'),
    ('iran',            'iran',              'xrRx85iA'),
    ('australia',       'australia',         'xSrf6qMM'),
    ('arabia_saudita',  'saudi-arabia',      'biSY8ox4'),
    ('uzbekistan',      'uzbekistan',        'EZYKKRMc'),
    ('iraq',            'iraq',              'K8aAGt6r'),
    ('jordania',        'jordan',            'vNcmJoU2'),

    ('qatar',           'qatar',             'zqzHL77i'),

    # CAF (10)
    ('marruecos',       'morocco',           'IDKYO3R8'),
    ('senegal',         'senegal',           'hOIsJLJr'),
    ('egipto',          'egypt',             'bejDn7NN'),

    ('argelia',         'algeria',           'nc87N1BR'),
    ('cabo_verde',      'cape-verde',        'MocyWdm7'),
    ('costa_marfil',    'ivory-coast',       'G2FRjBgn'),
    ('ghana',           'ghana',             'nNBjHale'),
    ('rd_congo',        'rd-congo',          'phn9mm8H'),
    ('sudafrica',       'south-africa',      'W2ijYvlr'),
    ('tunez',           'tunisia',           'QqZVYk95'),

    # OFC (1)
    ('nueva_zelanda',   'new-zealand',       'rLctHkpU'),
]

print(f'✅ {len(SELECCIONES)} selecciones cargadas')

print(f'✅ {len(SELECCIONES)} selecciones cargadas')
print('\nNOTA: Algunas selecciones tienen ID "UNKNOWN" — el scraper las detectará automáticamente.')

✅ 48 selecciones cargadas
✅ 48 selecciones cargadas

NOTA: Algunas selecciones tienen ID "UNKNOWN" — el scraper las detectará automáticamente.


## 🛠️ PASO 4: Funciones del scraper

Acá está toda la lógica. No necesitás modificar nada en esta celda.

In [ ]:
# ─── FUNCIÓN: Cargar/guardar progreso ─────────────────────────────────────────
def cargar_progreso():
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, 'r') as f:
            return json.load(f)
    return {'completados': [], 'errores': []}

def guardar_progreso(progreso):
    with open(LOG_FILE, 'w') as f:
        json.dump(progreso, f, indent=2)


# ─── FUNCIÓN: Aceptar cookies ─────────────────────────────────────────────────
async def aceptar_cookies(page):
    try:
        btn = page.locator('#onetrust-accept-btn-handler, button:has-text("Accept"), button:has-text("Aceptar")')
        if await btn.count() > 0:
            await btn.first.click()
            await asyncio.sleep(1)
    except:
        pass


# ─── FUNCIÓN: Obtener slug del partido desde los tab links ────────────────────
# Las URLs reales de Flashscore usan rutas de ruta, NO hashes.
# Ejemplo: /match/football/mexico-O6iHcNkd/serbia-8Kl6iq0i/summary/stats/?mid=b1x9Omno
# El "slug" es la parte /match/football/mexico-O6iHcNkd/serbia-8Kl6iq0i
# Lo extraemos del link "Stats" en la navegación secundaria del partido.
async def obtener_slug(page):
    try:
        tabs = page.locator('[data-testid="wcl-tabs"][data-type="secondary"] a')
        count = await tabs.count()
        for i in range(count):
            href = await tabs.nth(i).get_attribute('href')
            if href and '/summary/stats/' in href:
                return href.split('/summary/')[0]
    except:
        pass
    return None


# ─── FUNCIÓN: Obtener lista de partidos de una selección ──────────────────────
async def obtener_partidos(page, nombre_fs, id_fs):

    # Casos especiales de Flashscore
    if nombre_fs == "rd-congo":
        print("USANDO URL ESPECIAL RD CONGO")
        url = f"https://www.flashscore.com.mx/equipo/rd-congo/{id_fs}/resultados/"
    else:
        url = f"https://www.flashscore.com/team/{nombre_fs}/{id_fs}/results/"

    print(f'  🌐 Abriendo: {url}')

    await page.goto(url, wait_until='networkidle', timeout=60000)
    await aceptar_cookies(page)
    await asyncio.sleep(2)

    cantidad = 0

    for intento in range(20):
        partidos = page.locator('[class*="event__match"]')
        cantidad = await partidos.count()

        print(f'  📋 Partidos cargados: {cantidad}', end='\r')

        if cantidad >= PARTIDOS_POR_SELECCION:
            break

        btn_mas = page.locator(
            'a:has-text("Show more"), button:has-text("Show more")'
        )

        if await btn_mas.count() > 0:
            await btn_mas.first.click()
            await asyncio.sleep(2)
        else:
            break

    print(f'  📋 Partidos encontrados: {cantidad}')

    match_ids = []

    partidos_el = page.locator('[class*="event__match"]')

    for i in range(min(PARTIDOS_POR_SELECCION, await partidos_el.count())):
        try:
            el = partidos_el.nth(i)

            partido_id = await el.get_attribute('id')

            if partido_id:
                match_id = (
                    partido_id
                    .replace('g_1_', '')
                    .replace('g_2_', '')
                )

                match_ids.append(match_id)

        except Exception:
            pass

    return match_ids


# ─── FUNCIÓN: Extraer info básica del partido ─────────────────────────────────
async def extraer_info_basica(page):
    stats = {}
    try:
        stats['fecha'] = (await page.locator('.duelParticipant__startTime').first.text_content()).strip()
    except:
        stats['fecha'] = ''
    try:
        crumbs = page.locator('[data-testid="wcl-breadcrumbsItem"]')
        textos = []
        for i in range(await crumbs.count()):
            t = (await crumbs.nth(i).text_content()).strip()
            if t:
                textos.append(t)
        stats['competicion'] = ' / '.join(textos)
    except:
        stats['competicion'] = ''
    try:
        stats['equipo_local'] = (await page.locator(
            '.duelParticipant__home .participant__participantName'
        ).first.text_content()).strip()
    except:
        stats['equipo_local'] = ''
    try:
        stats['equipo_visitante'] = (await page.locator(
            '.duelParticipant__away .participant__participantName'
        ).first.text_content()).strip()
    except:
        stats['equipo_visitante'] = ''
    try:
        score_txt = (await page.locator('.detailScore__wrapper').first.text_content()).strip()
        nums = re.findall(r'\d+', score_txt)
        stats['goles_local']     = nums[0] if len(nums) > 0 else ''
        stats['goles_visitante'] = nums[1] if len(nums) > 1 else ''
    except:
        stats['goles_local']     = ''
        stats['goles_visitante'] = ''
    return stats


# ─── FUNCIÓN: Extraer filas de estadísticas de la página actual ───────────────
# Selectores confirmados con HTML real de Flashscore (data-testid estables):
#   [data-testid="wcl-statistics"]          → fila completa
#   [data-testid="wcl-statistics-category"] → nombre de la estadística
#   [data-testid="wcl-statistics-value"]    → valor (primer=local, segundo=visitante)
async def extraer_filas_stats(page, prefijo=''):
    resultado = {}
    try:
        await page.wait_for_selector('[data-testid="wcl-statistics"]', timeout=8000)
    except:
        return resultado

    filas = page.locator('[data-testid="wcl-statistics"]')
    n = await filas.count()
    for i in range(n):
        try:
            fila     = filas.nth(i)
            name_el  = fila.locator('[data-testid="wcl-statistics-category"]').first
            vals     = fila.locator('[data-testid="wcl-statistics-value"]')
            if await name_el.count() == 0:
                continue
            nombre   = (await name_el.text_content()).strip()
            nombre_k = re.sub(r'[^\w\s]', '', nombre, flags=re.UNICODE).strip().lower().replace(' ', '_')
            clave    = f'{prefijo}{nombre_k}'
            val_home = (await vals.nth(0).text_content()).strip() if await vals.count() > 0 else ''
            val_away = (await vals.nth(1).text_content()).strip() if await vals.count() > 1 else ''
            resultado[f'local__{clave}']     = val_home
            resultado[f'visitante__{clave}'] = val_away
        except:
            pass
    return resultado


# ─── HELPER: goto con reintentos y condición más flexible ─────────────────────
async def goto_seguro(page, url, retries=2):
    for intento in range(retries + 1):
        try:
            # domcontentloaded es mucho más rápido y confiable que networkidle
            await page.goto(url, wait_until='domcontentloaded', timeout=30000)
            await asyncio.sleep(2)  # Dar tiempo al JS para renderizar el contenido
            return True
        except PlaywrightTimeout:
            if intento < retries:
                print(f'     ⏳ Timeout en {url[:60]}... reintentando ({intento+1}/{retries})')
                await asyncio.sleep(5)
            else:
                return False
        except Exception as e:
            print(f'     ⚠️ Error navegando: {e}')
            return False
    return False


# ─── FUNCIÓN: Scrapear estadísticas generales ─────────────────────────────────
async def scrapear_estadisticas_generales(page, match_id):
    url_base = f'https://www.flashscore.com/match/{match_id}/'
    ok = await goto_seguro(page, url_base)
    if not ok:
        return {'match_id': match_id, 'error': 'timeout_base'}

    stats = await extraer_info_basica(page)
    stats['match_id'] = match_id

    slug = await obtener_slug(page)
    if not slug:
        stats['error'] = 'no_slug'
        return stats

    url_stats = f'https://www.flashscore.com{slug}/summary/stats/?mid={match_id}'
    ok = await goto_seguro(page, url_stats)
    if not ok:
        stats['error'] = 'timeout_stats'
        return stats  # Devuelve info básica aunque no haya stats

    PERIOD_KEYWORDS = ['full', '1st', '2nd', 'first', 'second', 'half',
                       'completo', 'primer', 'segundo', 'tiempo']
    all_tabs    = page.locator('[data-testid="wcl-tab"]')
    n_tabs      = await all_tabs.count()
    period_tabs = []
    for i in range(n_tabs):
        txt = (await all_tabs.nth(i).text_content()).strip()
        if any(kw in txt.lower() for kw in PERIOD_KEYWORDS):
            period_tabs.append((i, txt))

    if not period_tabs:
        filas = await extraer_filas_stats(page, prefijo='')
        stats.update(filas)
    else:
        for idx, lbl in period_tabs:
            prefijo = re.sub(r'[^\w\s]', '', lbl, flags=re.UNICODE).strip().lower().replace(' ', '_') + '__'
            try:
                await all_tabs.nth(idx).click()
                await asyncio.sleep(1.5)
                filas = await extraer_filas_stats(page, prefijo=prefijo)
                stats.update(filas)
            except Exception as e:
                stats[f'error_{lbl}'] = str(e)[:80]

    return stats


# ─── FUNCIÓN: Scrapear estadísticas de jugadores ──────────────────────────────
async def scrapear_estadisticas_jugadores(page, match_id, slug):
    jugadores = []
    if not slug:
        return jugadores

    url = f'https://www.flashscore.com{slug}/summary/player-stats/?mid={match_id}'
    ok = await goto_seguro(page, url)
    if not ok:
        return jugadores

    try:
        await page.wait_for_selector('table', timeout=8000)
    except:
        return jugadores

    html_parts = await page.evaluate("""() => {
        return Array.from(document.querySelectorAll('table')).map(t => t.outerHTML);
    }""")
    if not html_parts:
        return jugadores

    from bs4 import BeautifulSoup
    equipos = ['local', 'visitante']
    for t_idx, tabla_html in enumerate(html_parts[:2]):
        equipo = equipos[t_idx] if t_idx < len(equipos) else f'equipo_{t_idx}'
        soup   = BeautifulSoup(tabla_html, 'html.parser')
        tabla  = soup.find('table')
        if not tabla:
            continue
        headers = []
        thead = tabla.find('thead')
        if thead:
            ths = thead.find_all(['th', 'td'])
            headers = [
                re.sub(r'[^\w\s]', '', th.get_text(strip=True), flags=re.UNICODE)
                .strip().lower().replace(' ', '_') or f'col_{j}'
                for j, th in enumerate(ths)
            ]
        tbody = tabla.find('tbody') or tabla
        for fila in tbody.find_all('tr'):
            celdas = [td.get_text(strip=True) for td in fila.find_all(['td', 'th'])]
            if not any(celdas):
                continue
            jugador = {'match_id': match_id, 'equipo': equipo}
            for j, celda in enumerate(celdas):
                col = headers[j] if j < len(headers) else f'col_{j}'
                jugador[col] = celda
            if any(v for k, v in jugador.items() if k not in ('match_id', 'equipo') and v):
                jugadores.append(jugador)

    return jugadores

# ─── FUNCIÓN PRINCIPAL: Scrapear una selección ────────────────────────────────
async def scrapear_seleccion(browser, nombre_archivo, nombre_fs, id_fs):
    page = await browser.new_page(
        user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
    )
    todas_stats     = []
    todos_jugadores = []

    try:
        print(f'\n🔍 [{nombre_archivo.upper()}] Obteniendo lista de partidos...')
        match_ids = await obtener_partidos(page, nombre_fs, id_fs)
        print(f'  ✅ {len(match_ids)} partidos encontrados')

        for i, match_id in enumerate(match_ids, 1):
            print(f'  ⚽ Partido {i}/{len(match_ids)}: {match_id}')

            stats = await scrapear_estadisticas_generales(page, match_id)
            stats['seleccion'] = nombre_archivo

            # El slug sigue disponible en la última página cargada (stats tab)
            slug = await obtener_slug(page)
            todas_stats.append(stats)

            jugadores = await scrapear_estadisticas_jugadores(page, match_id, slug)
            for j in jugadores:
                j['seleccion'] = nombre_archivo
            todos_jugadores.extend(jugadores)

            campos = len([k for k in stats if k.startswith('local__')])
            print(f'     📊 Stats: {campos} campos | 👤 Jugadores: {len(jugadores)}')
            await asyncio.sleep(ESPERA_ENTRE_PARTIDOS)

        # ── CSV estadísticas generales ──
        if todas_stats:
            csv_general = f'{OUTPUT_DIR}/{nombre_archivo}_partidos.csv'
            keys = set()
            for s in todas_stats:
                keys.update(s.keys())
            prioridad = ['match_id', 'seleccion', 'fecha', 'competicion',
                         'equipo_local', 'equipo_visitante', 'goles_local', 'goles_visitante']
            resto    = sorted(k for k in keys if k not in prioridad)
            keys_ord = [k for k in prioridad if k in keys] + resto
            with open(csv_general, 'w', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=keys_ord, extrasaction='ignore')
                writer.writeheader()
                writer.writerows(todas_stats)
            print(f'  💾 {csv_general} → {len(todas_stats)} partidos, {len(keys_ord)} columnas')

        # ── CSV jugadores ──
        if todos_jugadores:
            csv_jugadores = f'{OUTPUT_DIR}/{nombre_archivo}_jugadores.csv'
            keys_j = set()
            for j in todos_jugadores:
                keys_j.update(j.keys())
            prioridad_j = ['match_id', 'seleccion', 'equipo']
            resto_j     = sorted(k for k in keys_j if k not in prioridad_j)
            keys_j_ord  = [k for k in prioridad_j if k in keys_j] + resto_j
            with open(csv_jugadores, 'w', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=keys_j_ord, extrasaction='ignore')
                writer.writeheader()
                writer.writerows(todos_jugadores)
            print(f'  💾 {csv_jugadores} → {len(todos_jugadores)} filas, {len(keys_j_ord)} columnas')
        else:
            print(f'  ⚠️  Sin datos de jugadores para {nombre_archivo}')

        return True

    except Exception as e:
        print(f'  ❌ Error en {nombre_archivo}: {e}')
        return False
    finally:
        await page.close()

print('✅ Funciones cargadas correctamente')


✅ Funciones cargadas correctamente


## ▶️ PASO 5: Ejecutar el scraper

Esta celda ejecuta el scraper. **Podés pausar y retomar** — el progreso se guarda automáticamente.

Si querés scrapear solo algunas selecciones, modificá `SELECCIONES_A_PROCESAR`.

In [ ]:
# ─── CONFIGURAR CUÁLES SELECCIONES PROCESAR ───────────────────────────────────
# Para todas las 48: SELECCIONES_A_PROCESAR = SELECCIONES
# Para probar con Argentina: SELECCIONES_A_PROCESAR = [SELECCIONES[0]]
# Para una región específica: SELECCIONES_A_PROCESAR = SELECCIONES[:6]  # CONMEBOL

#SELECCIONES_A_PROCESAR = SELECCIONES[:1]  # ← Empezar con Argentina para probar
SELECCIONES_A_PROCESAR = SELECCIONES  # ← Descomentar para todas

print(f'📋 Selecciones a procesar: {[s[0] for s in SELECCIONES_A_PROCESAR]}')


async def main():
    progreso = cargar_progreso()
    completados = set(progreso.get('completados', []))

    print(f'\n📊 Progreso anterior: {len(completados)} selecciones ya procesadas')
    if completados:
        print(f'   Ya completadas: {", ".join(sorted(completados))}')

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=['--no-sandbox', '--disable-setuid-sandbox', '--disable-dev-shm-usage']
        )

        for nombre_archivo, nombre_fs, id_fs in SELECCIONES_A_PROCESAR:
            if nombre_archivo in completados:
                print(f'⏭️  {nombre_archivo} ya procesado, saltando...')
                continue

            inicio = time.time()
            exito = await scrapear_seleccion(browser, nombre_archivo, nombre_fs, id_fs)
            duracion = time.time() - inicio

            if exito:
                progreso['completados'].append(nombre_archivo)
                print(f'  ⏱️  Tiempo: {duracion:.0f}s')
            else:
                progreso['errores'].append(nombre_archivo)

            guardar_progreso(progreso)
            await asyncio.sleep(ESPERA_ENTRE_SELECCIONES)

        await browser.close()

    print(f'\n🏁 Proceso completado!')
    print(f'   ✅ Completados: {len(progreso["completados"])}')
    print(f'   ❌ Con errores: {len(progreso["errores"])}')
    if progreso['errores']:
        print(f'   Errores en: {", ".join(progreso["errores"])}')
    print(f'   📁 Archivos en: {OUTPUT_DIR}')


await main()

📋 Selecciones a procesar: ['argentina', 'brasil', 'colombia', 'uruguay', 'ecuador', 'paraguay', 'alemania', 'espana', 'francia', 'portugal', 'inglaterra', 'belgica', 'netherlands', 'croacia', 'austria', 'suiza', 'escocia', 'turquia', 'republica_checa', 'bosnia', 'noruega', 'suecia', 'estados_unidos', 'mexico', 'canada', 'panama', 'curazao', 'haiti', 'japon', 'corea_sur', 'iran', 'australia', 'arabia_saudita', 'uzbekistan', 'iraq', 'jordania', 'qatar', 'marruecos', 'senegal', 'egipto', 'argelia', 'cabo_verde', 'costa_marfil', 'ghana', 'rd_congo', 'sudafrica', 'tunez', 'nueva_zelanda']

📊 Progreso anterior: 47 selecciones ya procesadas
   Ya completadas: alemania, arabia_saudita, argelia, argentina, australia, austria, belgica, bosnia, brasil, cabo_verde, canada, colombia, corea_sur, costa_marfil, croacia, curazao, ecuador, egipto, escocia, espana, estados_unidos, francia, ghana, haiti, inglaterra, iran, iraq, japon, jordania, marruecos, mexico, netherlands, noruega, nueva_zelanda, panam

## 📁 PASO 6: Ver archivos generados y descargarlos

In [ ]:
import os
import zipfile
from google.colab import files

# Listar archivos generados
archivos = sorted(os.listdir(OUTPUT_DIR))
print(f'📂 Archivos en {OUTPUT_DIR}:')
total_size = 0
for nombre in archivos:
    ruta = f'{OUTPUT_DIR}/{nombre}'
    size = os.path.getsize(ruta)
    total_size += size
    print(f'   {nombre} ({size/1024:.1f} KB)')

print(f'\n💾 Tamaño total: {total_size/1024/1024:.2f} MB')

# Comprimir todo en un ZIP para descargar fácilmente
zip_path = '/content/flashscore_mundial.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for nombre in archivos:
        zf.write(f'{OUTPUT_DIR}/{nombre}', nombre)

print(f'\n📦 ZIP creado: {zip_path} ({os.path.getsize(zip_path)/1024:.1f} KB)')
files.download(zip_path)
print('⬇️  Descarga iniciada!')

NameError: name 'OUTPUT_DIR' is not defined

## 🔍 PASO 7 (opcional): Vista previa de los datos

In [ ]:
import pandas as pd

# Cambiar 'argentina' por cualquier selección que hayas descargado
SELECCION_PREVIEW = 'argentina'

csv_partidos = f'{OUTPUT_DIR}/{SELECCION_PREVIEW}_partidos.csv'
csv_jugadores = f'{OUTPUT_DIR}/{SELECCION_PREVIEW}_jugadores.csv'

if os.path.exists(csv_partidos):
    df_partidos = pd.read_csv(csv_partidos)
    print(f'⚽ PARTIDOS de {SELECCION_PREVIEW.upper()}:')
    print(f'   Filas: {len(df_partidos)} | Columnas: {len(df_partidos.columns)}')
    print(f'   Columnas: {list(df_partidos.columns)[:10]}...')
    display(df_partidos.head(3))
else:
    print(f'❌ No existe el archivo para {SELECCION_PREVIEW}')

if os.path.exists(csv_jugadores):
    df_jugadores = pd.read_csv(csv_jugadores)
    print(f'\n👤 JUGADORES de {SELECCION_PREVIEW.upper()}:')
    print(f'   Filas: {len(df_jugadores)} | Columnas: {len(df_jugadores.columns)}')
    display(df_jugadores.head(3))
else:
    print(f'⚠️  No hay datos de jugadores para {SELECCION_PREVIEW}')

---
## ⚠️ Notas importantes

1. **IDs de Flashscore**: Si alguna selección no carga, el ID puede estar desactualizado. Buscá la selección en flashscore.com y copiá el ID de la URL (ej: `flashscore.com/team/nombre/XXXXXXXX/`).

2. **Bloqueos**: Flashscore puede bloquear si hacés demasiadas requests seguidas. El script tiene pausas para evitarlo.

3. **Sesión de Colab**: Colab desconecta después de ~90 min de inactividad. El progreso se guarda, así que podés retomar desde donde quedó.

4. **Estadísticas de jugadores**: No todos los partidos tienen estadísticas individuales en Flashscore (depende de la competición).

5. **Para procesar todas las 48**: En PASO 5, comentá la línea de `[:1]` y descomentá la de `SELECCIONES`.